In [1]:
import gradio as gr
import plotly.graph_objects as go
import pandas as pd
import requests
import html


def get_coordinates(city):

    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        if "results" not in data:
            return None, None, None

        result = data["results"][0]

        return (
            result["latitude"],
            result["longitude"],
            result.get("country", "")
        )

    except:
        return None, None, None


def weather_description(code):

    codes = {
        0: ("Clear Sky", "☀️"),
        1: ("Mainly Clear", "🌤️"),
        2: ("Partly Cloudy", "⛅"),
        3: ("Overcast", "☁️"),
        45: ("Fog", "🌫️"),
        48: ("Fog", "🌫️"),
        51: ("Light Drizzle", "🌦️"),
        53: ("Drizzle", "🌦️"),
        55: ("Heavy Drizzle", "🌧️"),
        61: ("Light Rain", "🌦️"),
        63: ("Moderate Rain", "🌧️"),
        65: ("Heavy Rain", "🌧️"),
        71: ("Light Snow", "🌨️"),
        73: ("Snow", "🌨️"),
        75: ("Heavy Snow", "❄️"),
        80: ("Rain Showers", "🌦️"),
        81: ("Rain Showers", "🌧️"),
        82: ("Heavy Rain Showers", "⛈️"),
        85: ("Snow Showers", "🌨️"),
        86: ("Heavy Snow", "❄️"),
        95: ("Thunderstorm", "⛈️"),
        96: ("Thunderstorm", "⛈️"),
        99: ("Thunderstorm", "⛈️")
    }

    return codes.get(
        code,
        ("Unknown", "🌡️")
    )


def create_cards(current):

    temperature = round(
        current["temperature_2m"], 1
    )

    humidity = round(
        current["relative_humidity_2m"]
    )

    wind = round(
        current["wind_speed_10m"], 1
    )

    rain = round(
        current["precipitation"], 1
    )

    pressure = round(
        current["surface_pressure"]
    )

    apparent = round(
        current["apparent_temperature"], 1
    )

    condition, icon = weather_description(
        current["weather_code"]
    )

    return f"""
    <div class="weather-grid">

        <div class="weather-card main-card">
            <div class="card-icon">{icon}</div>
            <div class="card-title">Temperature</div>
            <div class="card-value">{temperature}°C</div>
            <div class="card-sub">
                Feels like {apparent}°C
            </div>
        </div>

        <div class="weather-card">
            <div class="card-icon">💧</div>
            <div class="card-title">Humidity</div>
            <div class="card-value">{humidity}%</div>
            <div class="card-sub">
                Relative humidity
            </div>
        </div>

        <div class="weather-card">
            <div class="card-icon">💨</div>
            <div class="card-title">Wind</div>
            <div class="card-value">{wind}</div>
            <div class="card-sub">
                km/h
            </div>
        </div>

        <div class="weather-card">
            <div class="card-icon">🌧️</div>
            <div class="card-title">Precipitation</div>
            <div class="card-value">{rain}</div>
            <div class="card-sub">
                mm
            </div>
        </div>

        <div class="weather-card">
            <div class="card-icon">🔵</div>
            <div class="card-title">Pressure</div>
            <div class="card-value">{pressure}</div>
            <div class="card-sub">
                hPa
            </div>
        </div>

        <div class="weather-card">
            <div class="card-icon">☁️</div>
            <div class="card-title">Condition</div>
            <div class="card-value condition">
                {condition}
            </div>
            <div class="card-sub">
                Current weather
            </div>
        </div>

    </div>
    """


def create_forecast_chart(data):

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=data["Date"],
            y=data["Max"],
            mode="lines+markers",
            name="High",
            line=dict(
                width=3
            ),
            marker=dict(
                size=8
            )
        )
    )

    fig.add_trace(
        go.Scatter(
            x=data["Date"],
            y=data["Min"],
            mode="lines+markers",
            name="Low",
            line=dict(
                width=3
            ),
            marker=dict(
                size=8
            )
        )
    )

    fig.update_layout(
        title="7-Day Temperature Forecast",
        xaxis_title="",
        yaxis_title="Temperature °C",
        template="plotly_white",
        height=400,
        hovermode="x unified",
        margin=dict(
            l=30,
            r=30,
            t=60,
            b=30
        ),
        legend=dict(
            orientation="h",
            y=1.1,
            x=0
        )
    )

    return fig


def create_humidity_chart(data):

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=data["Date"],
            y=data["Humidity"],
            mode="lines+markers",
            name="Humidity",
            line=dict(
                width=3
            ),
            marker=dict(
                size=8
            ),
            fill="tozeroy"
        )
    )

    fig.update_layout(
        title="7-Day Humidity Forecast",
        xaxis_title="",
        yaxis_title="Humidity %",
        yaxis=dict(
            range=[0, 100]
        ),
        template="plotly_white",
        height=400,
        margin=dict(
            l=30,
            r=30,
            t=60,
            b=30
        )
    )

    return fig


def create_forecast_cards(data):

    output = '<div class="forecast-grid">'

    for _, row in data.iterrows():

        condition, icon = weather_description(
            row["Code"]
        )

        output += f"""

        <div class="day-card">

            <div class="day-name">
                {row["Day"]}
            </div>

            <div class="day-icon">
                {icon}
            </div>

            <div class="day-condition">
                {condition}
            </div>

            <div class="day-temp">
                {row["Max"]:.0f}° / {row["Min"]:.0f}°
            </div>

            <div class="day-info">
                💧 {row["Humidity"]:.0f}%
            </div>

            <div class="day-info">
                🌧️ {row["Rain"]:.0f}%
            </div>

            <div class="day-info">
                💨 {row["Wind"]:.0f} km/h
            </div>

        </div>

        """

    output += "</div>"

    return output


def get_weather(city):

    if not city:

        return (
            "Please enter a city.",
            "",
            go.Figure(),
            go.Figure()
        )

    latitude, longitude, country = get_coordinates(
        city
    )

    if latitude is None:

        return (
            "City not found. Please try another city.",
            "",
            go.Figure(),
            go.Figure()
        )

    url = "https://api.open-meteo.com/v1/forecast"

    params = {

        "latitude": latitude,

        "longitude": longitude,

        "current": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "apparent_temperature",
            "precipitation",
            "weather_code",
            "wind_speed_10m",
            "surface_pressure"
        ]),

        "daily": ",".join([
            "weather_code",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_probability_max",
            "precipitation_sum",
            "wind_speed_10m_max",
            "relative_humidity_2m_mean"
        ]),

        "forecast_days": 7,

        "timezone": "auto"
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=15
        )

        response.raise_for_status()

        result = response.json()

        current = result["current"]

        cards = create_cards(
            current
        )

        daily = result["daily"]

        dates = pd.to_datetime(
            daily["time"]
        )

        data = pd.DataFrame({

            "Date": dates,

            "Day": [
                date.strftime("%a")
                for date in dates
            ],

            "Max": daily[
                "temperature_2m_max"
            ],

            "Min": daily[
                "temperature_2m_min"
            ],

            "Humidity": daily[
                "relative_humidity_2m_mean"
            ],

            "Rain": daily[
                "precipitation_probability_max"
            ],

            "Wind": daily[
                "wind_speed_10m_max"
            ],

            "Code": daily[
                "weather_code"
            ]
        })

        temperature_chart = create_forecast_chart(
            data
        )

        humidity_chart = create_humidity_chart(
            data
        )

        forecast = create_forecast_cards(
            data
        )

        location = f"""
        <div class="location-header">

            <div class="location-icon">
                📍
            </div>

            <div>
                <div class="location-name">
                    {html.escape(city)}
                </div>

                <div class="location-country">
                    {html.escape(country)}
                </div>
            </div>

        </div>
        """

        return (
            location + cards,
            forecast,
            temperature_chart,
            humidity_chart
        )

    except Exception as e:

        return (
            f"""
            <div class="error">
                Unable to retrieve weather data.<br>
                {html.escape(str(e))}
            </div>
            """,
            "",
            go.Figure(),
            go.Figure()
        )


css = """

body {
    background: #f4f7fb;
}

.gradio-container {
    max-width: 1400px !important;
}

.header {
    text-align: center;
    padding: 20px 0 10px 0;
}

.title {
    font-size: 42px;
    font-weight: 800;
    letter-spacing: -1px;
}

.subtitle {
    color: #64748b;
    font-size: 16px;
    margin-top: 8px;
}

.search-box {
    background: white;
    padding: 18px;
    border-radius: 18px;
    box-shadow: 0 5px 20px rgba(0,0,0,0.06);
    margin: 15px 0;
}

.location-header {
    display: flex;
    align-items: center;
    gap: 15px;
    background: white;
    padding: 18px 22px;
    border-radius: 18px;
    margin: 15px 0;
    box-shadow: 0 5px 20px rgba(0,0,0,0.06);
}

.location-icon {
    font-size: 32px;
}

.location-name {
    font-size: 27px;
    font-weight: 800;
}

.location-country {
    color: #64748b;
    font-size: 14px;
    margin-top: 3px;
}

.weather-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 16px;
    margin: 18px 0 25px 0;
}

.weather-card {
    background: white;
    padding: 24px;
    border-radius: 20px;
    text-align: center;
    box-shadow: 0 5px 20px rgba(0,0,0,0.06);
    transition: transform 0.2s;
}

.weather-card:hover {
    transform: translateY(-3px);
}

.card-icon {
    font-size: 35px;
    margin-bottom: 8px;
}

.card-title {
    color: #64748b;
    font-size: 14px;
}

.card-value {
    font-size: 28px;
    font-weight: 800;
    margin-top: 8px;
}

.card-sub {
    color: #94a3b8;
    font-size: 12px;
    margin-top: 5px;
}

.condition {
    font-size: 20px;
}

.forecast-grid {
    display: grid;
    grid-template-columns: repeat(7, 1fr);
    gap: 12px;
    margin: 15px 0 30px 0;
}

.day-card {
    background: white;
    border-radius: 18px;
    padding: 18px 10px;
    text-align: center;
    box-shadow: 0 5px 18px rgba(0,0,0,0.06);
}

.day-name {
    font-size: 16px;
    font-weight: 800;
}

.day-icon {
    font-size: 38px;
    margin: 12px 0 6px 0;
}

.day-condition {
    color: #64748b;
    font-size: 12px;
    min-height: 32px;
}

.day-temp {
    font-size: 18px;
    font-weight: 800;
    margin: 10px 0;
}

.day-info {
    font-size: 12px;
    color: #64748b;
    margin-top: 5px;
}

.section-title {
    font-size: 23px;
    font-weight: 800;
    margin-top: 25px;
}

.error {
    background: #fee2e2;
    color: #991b1b;
    padding: 20px;
    border-radius: 15px;
    margin: 15px 0;
}

.footer {
    text-align: center;
    color: #94a3b8;
    font-size: 12px;
    padding: 25px;
}

@media(max-width: 900px) {

    .weather-grid {
        grid-template-columns: repeat(2, 1fr);
    }

    .forecast-grid {
        grid-template-columns: repeat(2, 1fr);
    }

}

"""


with gr.Blocks(
    title="Weather Intelligence Dashboard"
) as demo:

    gr.HTML(
        """
        <div class="header">

            <div class="title">
                🌤️ Weather Intelligence
            </div>

            <div class="subtitle">
                Real-time weather conditions and
                7-day forecast
            </div>

        </div>
        """
    )

    with gr.Row(
        elem_classes="search-box"
    ):

        city = gr.Textbox(
            value="Kolkata",
            label="Search City",
            placeholder="Enter city name..."
        )

        search = gr.Button(
            "🔍 Search Weather",
            variant="primary"
        )

    location_output = gr.HTML()

    gr.Markdown(
        "## Current Conditions"
    )

    weather_output = gr.HTML()

    gr.Markdown(
        "## 7-Day Forecast"
    )

    forecast_output = gr.HTML()

    with gr.Row():

        temperature_chart = gr.Plot(
            label="Temperature"
        )

        humidity_chart = gr.Plot(
            label="Humidity"
        )

    gr.HTML(
        """
        <div class="footer">
            Weather data powered by Open-Meteo
        </div>
        """
    )

    search.click(
        fn=get_weather,
        inputs=city,
        outputs=[
            location_output,
            forecast_output,
            temperature_chart,
            humidity_chart
        ]
    )

    # Moved demo.load() inside the gr.Blocks context
    demo.load(
        fn=get_weather,
        inputs=city,
        outputs=[
            location_output,
            forecast_output,
            temperature_chart,
            humidity_chart
        ]
    )


demo.launch(
    share=True,
    inline=True,
    css=css
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cd1106791dcdceff3d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
